<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/Marvin/dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datenpipeline für Sprachmodelle: BPE Tokenizer, Embeddings & DataLoader

**Autor:** Marvin Ebi · **Datum:** 12.10.2026

**Ziel:** Dieses Notebook zeigt die Erstellung einer Datenpipeline für Sprachmodelle in PyTorch. Es umfasst die Implementierung eines gleitenden Textfenster-Datasets (Sliding Window) mit Byte-Pair-Encoding (BPE) Tokenisierung, die Erstellung von Token- und Positions-Embeddings sowie einen flexiblen DataLoader und einen einfachen Trainings- und Validierungsdurchlauf.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken
from typing import Tuple

In [ ]:
# ============================================================================
# Dataset mit BPE Tokenization
# ============================================================================
class BPEDataset(Dataset):
    """Sliding Window Dataset mit BPE (Byte-Pair-Encoding) Tokenisierung.

    Dieses Dataset segmentiert einen Fließtext mittels eines gleitenden Fensters
    in Sequenzen fester Länge für das Training von autoregressiven Sprachmodellen.
    """

    def __init__(self, text: str, seq_len: int = 128, stride: int = 1,
                 encoding: str = "gpt2"):
        """Initialisiert das BPEDataset.

        Args:
            text (str): Der Eingabetext, der tokenisiert und geladen werden soll.
            seq_len (int): Die Länge der Token-Sequenz pro Sample (inklusive Target).
                           Standardwert ist 128.
            stride (int): Die Schrittweite des gleitenden Fensters. Standardwert ist 1.
            encoding (str): Name des tiktoken BPE-Encodings. Standardwert ist "gpt2".
        """
        self.tokenizer = tiktoken.get_encoding(encoding)
        self.tokens = self.tokenizer.encode(text)
        self.seq_len = seq_len
        self.stride = stride
        self.n_samples = max(0, (len(self.tokens) - seq_len) // stride + 1)

    def __len__(self) -> int:
        """Gibt die Anzahl der verfügbaren Samples im Dataset zurück.

        Returns:
            int: Anzahl der Samples.
        """
        return self.n_samples

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Erzeugt ein einzelnes Datenpaar (Input, Target) für den gegebenen Index.

        Das Target entspricht dem um eins nach rechts verschobenen Input-Fenster.

        Args:
            idx (int): Der Index des abzufragenden Samples.

        Returns:
            Tuple[torch.Tensor, torch.Tensor]: Ein Tupel bestehend aus:
                - Input-Token-IDs (Tensor der Form [seq_len - 1])
                - Target-Token-IDs (Tensor der Form [seq_len - 1])
        """
        start = idx * self.stride
        seq = self.tokens[start:start + self.seq_len]
        return torch.tensor(seq[:-1]), torch.tensor(seq[1:])

    def decode(self, tokens) -> str:
        """Dekodiert eine Liste oder einen Tensor von Token-IDs zurück in Text.

        Args:
            tokens (list, tuple or torch.Tensor): Die zu dekodierenden Token-IDs.

        Returns:
            str: Der dekodierte Text.
        """
        if isinstance(tokens, torch.Tensor):
            tokens = tokens.tolist()
        return self.tokenizer.decode(tokens)

    @property
    def vocab_size(self) -> int:
        """Gibt die Größe des Vokabulars des verwendeten Tokenizers zurück.

        Returns:
            int: Vokabulargröße.
        """
        return self.tokenizer.n_vocab

In [ ]:
# ============================================================================
# Embedding Layer
# ============================================================================
class Embeddings(nn.Module):
    """Kombinierter Embedding-Layer für Token- und Positions-Repräsentationen.

    Addiert die gelernten Einbettungen der Token-IDs mit den gelernten
    absoluten Positions-Einbettungen.
    """

    def __init__(self, vocab_size: int, emb_dim: int, max_len: int = 512):
        """Initialisiert die Embedding-Schichten.

        Args:
            vocab_size (int): Die Größe des Token-Vokabulars.
            emb_dim (int): Die Dimensionalität des Einbettungsraums (Embedding-Dimension).
            max_len (int): Die maximal unterstützte Sequenzlänge für Positions-Embeddings.
                           Standardwert ist 512.
        """
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(max_len, emb_dim)
        nn.init.normal_(self.token_emb.weight, 0, 0.02)
        nn.init.normal_(self.pos_emb.weight, 0, 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Führt den Forward-Pass der Einbettungen aus.

        Args:
            x (torch.Tensor): Tensor mit Token-IDs der Form [Batch-Größe, Sequenzlänge].

        Returns:
            torch.Tensor: Summiertes Token- und Positions-Embedding der Form
                          [Batch-Größe, Sequenzlänge, Embedding-Dimension].
        """
        batch, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).expand(batch, -1)
        return self.token_emb(x) + self.pos_emb(positions)

In [ ]:
# ============================================================================
# DataLoader Factory
# ============================================================================
def create_dataloader(text: str, seq_len: int = 128, batch_size: int = 32,
                      stride: int = 1, emb_dim: int = 256, shuffle: bool = True):
    """Erstellt Dataset, DataLoader und Embeddings in einem Schritt.

    Bequeme Hilfsfunktion, um schnell alle notwendigen Komponenten für eine
    Eingabepipeline aufzusetzen.

    Args:
        text (str): Der Rohtext, der für das Dataset herangezogen wird.
        seq_len (int): Die gewünschte Sequenzlänge pro Datenpunkt. Standardwert ist 128.
        batch_size (int): Die Batch-Größe für den DataLoader. Standardwert ist 32.
        stride (int): Die Schrittweite beim Extrahieren der Sequenzen. Standardwert ist 1.
        emb_dim (int): Die Dimension der generierten Embeddings. Standardwert ist 256.
        shuffle (bool): Bestimmt, ob die Daten im DataLoader gemischt werden. Standardwert ist True.

    Returns:
        Tuple[DataLoader, Embeddings, BPEDataset]: Ein Tupel bestehend aus:
            - DataLoader: PyTorch DataLoader über dem BPEDataset.
            - Embeddings: Instanziiertes Embedding-Modul passend zum Vokabular.
            - BPEDataset: Das erzeugte Dataset-Objekt.
    """
    dataset = BPEDataset(text, seq_len=seq_len, stride=stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    embeddings = Embeddings(dataset.vocab_size, emb_dim=emb_dim, max_len=seq_len)

    print(f"DataLoader ready: {len(dataset)} samples, {len(dataloader)} batches")
    return dataloader, embeddings, dataset

## Tests

In [ ]:
# ============================================================================
# Batch Inspection
# ============================================================================
def inspect_batch(dataloader, embeddings, dataset, n_batches=2):
    """Inspect batches with decoded text."""

    for i, (input_ids, target_ids) in enumerate(dataloader):
        if i >= n_batches:
            break

        emb = embeddings(input_ids)

        print(f"\n{'='*50}")
        print(f"BATCH {i+1} | Input: {input_ids.shape} | Emb: {emb.shape}")
        print(f"{'='*50}")
        print(f"Input:  '{dataset.decode(input_ids[0][:15])}'")
        print(f"Target: '{dataset.decode(target_ids[0][:15])}'")
        print(f"Emb stats: mean={emb.mean():.4f}, std={emb.std():.4f}")

In [ ]:
# ============================================================================
# Mini Training Loop
# ============================================================================
def train_step(dataloader, embeddings, steps=5, lr=0.001):
    """Simple training loop for testing."""

    vocab_size = embeddings.token_emb.num_embeddings
    emb_dim = embeddings.token_emb.embedding_dim

    # Simple LM head
    head = nn.Linear(emb_dim, vocab_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(list(embeddings.parameters()) + list(head.parameters()), lr=lr)

    print(f"\n{'='*50}")
    print("TRAINING")
    print(f"{'='*50}")

    for step in range(steps):
        for input_ids, target_ids in dataloader:
            emb = embeddings(input_ids)
            logits = head(emb)
            loss = criterion(logits.view(-1, vocab_size), target_ids.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(f"Step {step+1}: Loss = {loss.item():.4f}")
            break  # One batch per step

    return embeddings, head

In [ ]:
# Test BPEDataset
text = "Artificial intelligence is transforming the world. " * 10
dataset = BPEDataset(text, seq_len=32, stride=8)
print(f"Dataset: {len(dataset)} samples, vocab={dataset.vocab_size}")
print(f"Sample: {dataset.decode(dataset[0][0][:10])}")

In [ ]:
# Test Embedding Layer
emb = Embeddings(dataset.vocab_size, emb_dim=128, max_len=32)
sample_input, _ = dataset[0]
output = emb(sample_input.unsqueeze(0))
print(f"Embeddings: {sample_input.shape} -> {output.shape}")

In [ ]:
# Test Data Loader factory
dataloader, embeddings, dataset = create_dataloader(
    text=text, seq_len=32, batch_size=4, emb_dim=128
)

In [ ]:
# Batch Inspection
inspect_batch(dataloader, embeddings, dataset)

In [ ]:
# Test Training Loop
embeddings, head = train_step(dataloader, embeddings, steps=3)

In [ ]:
# ============================================================================
# Usage
# ============================================================================
"""
QUICK START
"""

# 1. Create (Verwende einen längeren Text oder eine kleinere seq_len, damit n_samples > 0 ist)
dl, emb, ds = create_dataloader("Artificial intelligence is transforming the world.", seq_len=4, batch_size=2)

# 2. Get batch
input_ids, target_ids = next(iter(dl))

# 3. Get embeddings
embeddings = emb(input_ids)

print("Ready to use!")